# 04 — Ensemble Evaluation

Compare CNN14 / AST weight choices and evaluate the final test split.

## Phase 7 Results (from kit-pri-v2.ipynb, Kaggle, 2026-06-09)

| Metric | Val | Test |
|---|---:|---:|
| Best Val F1  | **0.9420** (epoch 35) | — |
| Test F1      | —  | **0.9275** |
| Test AUC     | —  | 0.9573 |
| Test Accuracy| —  | 0.9271 |

Threshold sweep (on test, after training): best F1 at thresh=0.60 → F1=0.9292.

> **Important**: threshold must be selected on validation data only, never on test.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from src.utils import binary_metrics, find_optimal_threshold

# ─── Replace these with saved .npy logit files from your checkpoint ──────────
# val_probs  = np.load('results/val_probs.npy')
# val_labels = np.load('results/val_labels.npy')
# test_probs  = np.load('results/test_probs.npy')
# test_labels = np.load('results/test_labels.npy')

# Demo with synthetic data (replace when checkpoint is available)
rng = np.random.default_rng(42)
N_val, N_test = 531, 549
val_labels  = rng.integers(0, 2, N_val)
val_probs   = np.clip(val_labels * 0.7 + rng.normal(0, 0.2, N_val), 0, 1)
test_labels = rng.integers(0, 2, N_test)
test_probs  = np.clip(test_labels * 0.7 + rng.normal(0, 0.2, N_test), 0, 1)

print('Data shapes:', val_probs.shape, test_probs.shape)

In [ ]:
# ─── Threshold selection on VALIDATION data only ─────────────────────────────
best_thresh, best_val_f1 = find_optimal_threshold(
    val_labels, val_probs, minimum=0.30, maximum=0.70, step=0.01
)
print(f'Best threshold (val): {best_thresh:.2f}  ->  Val F1={best_val_f1:.4f}')

# Evaluate on test with the val-selected threshold
test_metrics = binary_metrics(test_labels, test_probs, threshold=best_thresh)
print(f'\nTest metrics @ threshold={best_thresh}:')
for k, v in test_metrics.items():
    print(f'  {k:<22}: {v}')

In [ ]:
# ─── CNN14 / AST weight sweep (on validation) ────────────────────────────────
# Replace with branch logits from saved checkpoint
# cnn14_logits_val = ...
# ast_logits_val   = ...

# Placeholder sweep showing the search structure
ast_weights = np.arange(0.0, 1.01, 0.05)
f1_scores   = []
for w_ast in ast_weights:
    w_cnn14 = 1.0 - w_ast
    # combined = torch.sigmoid(w_cnn14 * cnn14_logits + w_ast * ast_logits)
    combined = val_probs  # placeholder
    f1_scores.append(f1_score(val_labels, combined >= 0.5))

plt.figure(figsize=(8, 4))
plt.plot(ast_weights, f1_scores, marker='o', color='#1565C0')
plt.axvline(0.55, color='red', linestyle='--', label='Notebook best: AST=0.55')
plt.xlabel('AST weight (CNN14 weight = 1 - AST)')
plt.ylabel('Validation F1')
plt.title('Ensemble weight sweep — select on validation, apply to test')
plt.legend()
plt.tight_layout()
plt.show()
print('Notebook final weights: CNN14=0.45  AST=0.55')

In [ ]:
# ─── Confusion matrix ─────────────────────────────────────────────────────────
import seaborn as sns

cm = np.array(test_metrics['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted NC', 'Predicted C'],
    yticklabels=['True NC', 'True C'],
    ax=ax
)
ax.set_title(f'Confusion Matrix (thresh={best_thresh:.2f})')
plt.tight_layout()
plt.show()